In [1]:
import lmstudio as lms
import pandas as pd
import os
from PIL import Image
from sklearn.metrics import *
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import time
from colorama import Fore, Style
import re
import numpy as np

In [2]:
os.environ["CUDA_VISIBLE_DEVICES"] = "1" # Only use the 2nd GPU

In [3]:
# Qwen
#MODEL_NAME = "qwen/qwen2.5-vl-7b"
#MODEL_NAME = "qwen/qwen3-vl-4b"
#MODEL_NAME = "qwen/qwen3-vl-8b"
#MODEL_NAME = "qwen/qwen3-vl-30b"

# Gemma
#MODEL_NAME = "google/gemma-3-4b"
#MODEL_NAME = "google/gemma-3-12b"
#MODEL_NAME = "google/gemma-3-27b"

#MODEL_NAME = "gemma-4-26b-a4b-it"
#MODEL_NAME = "gemma-4-e2b-it"
MODEL_NAME = "gemma-4-e4b-it"

# LMF2 LiquidAI
#MODEL_NAME = "liquidai_lfm2-vl-450m"
#MODEL_NAME = "lfm2-vl-1.6b"
#MODEL_NAME = "lfm2-vl-3b"

# MistralAI
#MODEL_NAME = "mistralai/ministral-3-3b"
#MODEL_NAME = "mistralai/magistral-small-2509" # This is a reasoning model, which is expected to be slower

model = lms.llm(MODEL_NAME)

In [4]:
# Example

sample_img = "covid_memes_5433.png" # Actual label is 1

In [5]:
# Setup file paths
IMG_DIR   = "../../HarMeme-C_Dataset/images/"

test_dataset = "../../HarMeme-C_Dataset/test_bin.csv"


In [6]:
def strip_thinking(text: str) -> str:
    # Remove any reasoning or thinking blocks, case-insensitive
    cleaned = re.sub(
        r"[\[<]\s*think\s*[\]>].*?[\[<]\s*/\s*think\s*[\]>]",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    )
    return cleaned.strip()

In [7]:
# Warm up

result = model.respond("Hi how are you")
print(result)

I'm doing well, thank you! I hope you are having a good day too.

How can I help you today? 😊


In [8]:
result = str(result)
print(strip_thinking(result))

I'm doing well, thank you! I hope you are having a good day too.

How can I help you today? 😊


In [9]:
example_path = os.path.join(IMG_DIR, str(sample_img))
example_img_handle = lms.prepare_image(example_path)

In [10]:
# Warming up with image input
# Actual / Correct label is 1

for i in range(5):
    chat = lms.Chat()

    chat.add_user_message(
        "Reply with ONLY 1 for Yes OR 0 for NO. Nothing else. Do not describe the image. The question being: is this image considered hate speech or offensive?",
        images=[example_img_handle]
    )
    example_response = str(model.respond(chat))
    example_response = strip_thinking(example_response) 
    print(f"Response no. {i+1}\t: {example_response}")

    del chat

Response no. 1	: 0
Response no. 2	: 0
Response no. 3	: 0
Response no. 4	: 0
Response no. 5	: 0


In [11]:
#model.unload() 

In [12]:
# Setup dataset
#df = pd.read_csv(dataset) # Entire dataset
df = pd.read_csv(test_dataset) # Test dataset
display(df) # before
# Label encoding
#label_encoder = LabelEncoder()
df['label'] = df['label'].tolist()
filenames = df['image'].tolist()
actual_labels = df['label'].tolist()

,image,text,label
0,covid_memes_5425.png,gwen\n@gwenervi\ndis gon be trump tomorrow aft...,0
1,covid_memes_5426.png,Armani\n@historyofarmani\nBiden after hearing ...,0
2,covid_memes_5429.png,MESSAGE FROM TRUMP TO\nCOVID-19\nLEAVE NOW OR ...,0
3,covid_memes_5430.png,COVID-19 STARTED DURING HIS TERM\nSO IT SHOULD...,0
4,covid_memes_5434.png,TRUMPS RESPONSE TO COVID-19\nUMP\nTAYM\nINCLUD...,0
...,...,...,...
349,covid_memes_2041.png,ONE DAY IN THE FUTURE\n,0
350,covid_memes_2049.png,IndieWire\nIndieWire\n@quentquarantino\n@Indie...,0
351,covid_memes_2058.png,450/4 steady batting still have\nanother 10 da...,0
352,covid_memes_2062.png,1\n200\n20\nMemes on Baba Ramdev's anti Covid-...,0


In [13]:
#display(df) # after

In [14]:
# Set the batch size to fit the token limit
batch_size = 10

# Split the filenames into batches
batches = [filenames[i:i + batch_size] for i in range(0, len(filenames), batch_size)]

label_map = dict(zip(df['image'], df['label'])) # Map filenames to actual labels

safe_model_name = re.sub(r'[\\/*?:"<>|/]', '_', MODEL_NAME)

evaluation_file_name = f"{safe_model_name}_results.csv"

processed_images = set()
results_list = []
predictions = []

In [15]:
# Loop over each batch of images

# Start Timer
start_time = time.perf_counter()

# Calculate total for the progress bar
total_images = 354 
pbar = tqdm(total=total_images, desc="Processing Images", unit="img")

for batch in batches:
    try:
        for filename in batch:
            # Reset chat for each batch, reset again per image, ensuring a clean start
            chat = lms.Chat()
            if filename in processed_images:
                continue

            image_path = os.path.join(IMG_DIR, filename)
            image_handle = lms.prepare_image(image_path)

            processed_images.add(filename)

            chat.add_user_message(
                "Reply with ONLY 1 for Yes OR 0 for NO. Nothing else. Do not describe the image. The question being: is this image considered hate speech or offensive?",
                images=[image_handle]
            )

            # Get the response for the single image immediately
            response = str(model.respond(chat))
            response = strip_thinking(response)

            # Normalize and handle the single response immediately
            single_prediction = -1  # Use -1 as a placeholder for failed predictions
            
            if isinstance(response, list):
                if response and response[0].strip() in ["0", "1"]:
                    single_prediction = int(response[0].strip())
            elif response in ["0", "1"]:
                single_prediction = int(response)

            # Collect and store the result
            predictions.append(single_prediction)  # Keep for overall metrics

            actual_label = label_map.get(filename, 'N/A')  # Look up the true label

            results_list.append({
                'filename': filename,
                'actual_label': actual_label,
                'predicted_label': single_prediction
            })

            # Update the progress bar by 1
            pbar.update(1)

        # Reset the chat history for the next batch after processing all images
        chat = lms.Chat()  # Reset chat here if needed, after all images in the batch

    except Exception as e:
        print(f"{Style.BRIGHT}{Fore.RED}Error occurred during batch processing: {e}{Style.RESET_ALL}")
        model.unload()
        break

pbar.close() # Close tqdm bar

# End Timer
end_time = time.perf_counter()
elapsed_time = end_time - start_time

# Create the final DataFrame from the list of results
results_df = pd.DataFrame(results_list)

# Display the head of the results DataFrame
print("\n--- Results DataFrame Head ---")
display(results_df.head())
print("----------------------------\n")

# Print Elapsed Time 
hours, remainder = divmod(elapsed_time, 3600)
minutes, seconds = divmod(remainder, 60)
print(f"{Style.BRIGHT}{Fore.MAGENTA}Total Evaluation Time: {int(hours):02d}h {int(minutes):02d}m {seconds:05.2f}s{Style.RESET_ALL}")

# Final accuracy calculation
# Filter out samples where prediction failed (-1) before calculating accuracy
valid_results_df = results_df[results_df['predicted_label'].isin([0, 1])].copy()
valid_predictions = valid_results_df['predicted_label'].tolist()
valid_actuals = valid_results_df['actual_label'].tolist()

Processing Images: 100%|██████████| 354/354 [19:09<00:00,  3.25s/img]


--- Results DataFrame Head ---


,filename,actual_label,predicted_label
0,covid_memes_5425.png,0,0
1,covid_memes_5426.png,0,0
2,covid_memes_5429.png,0,0
3,covid_memes_5430.png,0,0
4,covid_memes_5434.png,0,0


----------------------------

Total Evaluation Time: 00h 19m 09.86s


In [16]:
print(f"{Style.BRIGHT}{Fore.BLUE}Total processed images: {len(results_df)}{Style.RESET_ALL}")
print(f"{Style.BRIGHT}{Fore.GREEN}Valid predictions used for metrics: {len(valid_predictions)}{Style.RESET_ALL}")
time_str = f"{int(hours):02d}:{int(minutes):02d}:{int(seconds):02d}" 
print(f"{Style.BRIGHT}{Fore.MAGENTA}Total Evaluation Time: {time_str}{Style.RESET_ALL}")


if len(valid_predictions) > 0:
    # Prepare the list for corrected predictions, treating -1 as incorrect
    results_df['predicted_label_corrected'] = results_df['predicted_label'].apply(lambda x: x if x != -1 else None)

    # Filter out rows where predicted_label is None (i.e., -1 values)
    valid_results_df = results_df[results_df['predicted_label_corrected'].notna()]

    valid_predictions = valid_results_df['predicted_label_corrected'].tolist()
    valid_actuals = valid_results_df['actual_label'].tolist()

    # Calculate metrics, treating -1 as incorrect but keeping 0 as valid if it matches the actual label
    accuracy = np.round(accuracy_score(valid_actuals, valid_predictions), 4)
    precision, recall, f1, _ = precision_recall_fscore_support(
        valid_actuals, valid_predictions, labels=None, pos_label=1, average=None, zero_division=0, beta=1.0
    )
    mcc = np.round(matthews_corrcoef(valid_actuals, valid_predictions), 4)
    precision = np.round(precision, 4)
    recall = np.round(recall, 4)
    f1 = np.round(f1, 4)

    # Calculate correct predictions
    correct_predictions = sum(1 for p, a in zip(valid_predictions, valid_actuals) if p == a)

    # Print the metrics
    print(f"\n--- {Style.BRIGHT}{Fore.YELLOW}{MODEL_NAME} Performance Metrics ---{Style.RESET_ALL}")
    print(f"{Style.BRIGHT}{Fore.GREEN}Accuracy\t: {accuracy:.4f}{Style.RESET_ALL}")
    print(f"{Style.BRIGHT}{Fore.GREEN}Precision\t: {', '.join(f'{p:.4f}' for p in precision)}{Style.RESET_ALL} ")
    print(f"{Style.BRIGHT}{Fore.GREEN}Recall\t\t: {', '.join(f'{r:.4f}' for r in recall)}{Style.RESET_ALL} ")
    print(f"{Style.BRIGHT}{Fore.GREEN}F1 Score\t: {', '.join(f'{f1_score:.4f}' for f1_score in f1)}{Style.RESET_ALL} ")
    print(f"{Style.BRIGHT}{Fore.GREEN}MCC\t\t: {mcc:.4f}{Style.RESET_ALL}")
    print(f"{Style.BRIGHT}{Fore.YELLOW}{'-'*50}\n{Style.RESET_ALL}")
    if len(valid_predictions) == total_images:
        print(f"{Style.BRIGHT}{Fore.GREEN}Out of {len(valid_predictions)} valid predictions, correctly predicted {correct_predictions} labels{Style.RESET_ALL}")
    else:
        print(f"{Style.BRIGHT}{Fore.RED}Out of {len(valid_predictions)} valid predictions, correctly predicted {correct_predictions} labels{Style.RESET_ALL}")   
    print(f"{Style.BRIGHT}{Fore.CYAN}\nClassification Report:\n\n{classification_report(valid_actuals, valid_predictions, digits=4)}{Style.RESET_ALL}")
else:
    print(f"{Style.BRIGHT}{Fore.RED}No valid predictions (0 or 1) were collected for metrics calculation.{Style.RESET_ALL}")

Total processed images: 354
Valid predictions used for metrics: 354
Total Evaluation Time: 00:19:09

--- gemma-4-e4b-it Performance Metrics ---
Accuracy	: 0.6554
Precision	: 0.6570, 0.6000 
Recall		: 0.9826, 0.0484 
F1 Score	: 0.7875, 0.0896 
MCC		: 0.0892
--------------------------------------------------

Out of 354 valid predictions, correctly predicted 232 labels

Classification Report:

              precision    recall  f1-score   support

           0     0.6570    0.9826    0.7875       230
           1     0.6000    0.0484    0.0896       124

    accuracy                         0.6554       354
   macro avg     0.6285    0.5155    0.4385       354
weighted avg     0.6370    0.6554    0.5430       354



In [17]:
display(results_df)

,filename,actual_label,predicted_label,predicted_label_corrected
0,covid_memes_5425.png,0,0,0
1,covid_memes_5426.png,0,0,0
2,covid_memes_5429.png,0,0,0
3,covid_memes_5430.png,0,0,0
4,covid_memes_5434.png,0,0,0
...,...,...,...,...
349,covid_memes_2041.png,0,0,0
350,covid_memes_2049.png,0,0,0
351,covid_memes_2058.png,0,0,0
352,covid_memes_2062.png,0,0,0


In [18]:
model.unload() # Eject the model from memory
print("Model unloaded")

Model unloaded
